# Titanic Random Forest

## Project overview


### Objective

The objective of this project is to implement a Random Forest classifier from scratch to predict passenger survival on the Titanic dataset. Rather than relying on pre-built machine learning implementations, the Decision Tree and Random Forest algorithms are developed manually to gain a deeper understanding of ensemble learning and tree-based classification.

This project follows a complete machine learning workflow, including data exploration, preprocessing, model implementation, evaluation, and analysis.


### Dataset
This project uses the **Titanic – Machine Learning from Disaster** dataset provided by Kaggle.

**Problem Type:** Binary Classification

**Target Variable:** `Survived`

- `0` → Passenger did not survive
- `1` → Passenger survived



### Project Goals

The primary goals of this project are:

- Perform exploratory data analysis (EDA) to understand the dataset.
- Handle missing values and prepare the data for machine learning.
- Engineer meaningful features from the raw dataset.
- Implement a Decision Tree classifier from scratch.
- Build a Random Forest classifier using multiple Decision Trees.
- Evaluate model performance using manually implemented evaluation metrics.
- Analyze feature importance and interpret the model's predictions.

### Learning Objectives

This project is intended to develop a strong understanding of the algorithms behind Random Forests rather than simply using existing implementations.

The concepts explored include:

- Decision Tree learning
- Recursive tree construction
- Gini Impurity
- Information Gain
- Feature selection for node splitting
- Bootstrap sampling (Bagging)
- Random feature selection
- Majority voting
- Ensemble learning
- Model evaluation metrics


### Technologies Used

Programming Language

- Python

Libraries

- NumPy
- Pandas
- Matplotlib
- Seaborn

Development Environment

- Google Colab

Version Control

- Git & GitHub

### Project Structure

The notebook is organized into the following sections:

1. Project Overview
2. Import Libraries
3. Data Loading
4. Exploratory Data Analysis (EDA)
5. Data Preprocessing
6. Decision Tree Implementation
7. Random Forest Implementation
8. Hyperparameter Tuning
9. Model Evaluation
10. Feature Importance Analysis
11. Results and Discussion
12. Conclusion


### Implementation Constraints

To ensure a thorough understanding of ensemble learning, the following constraints are followed throughout this project:

- No use of `sklearn.ensemble.RandomForestClassifier`
- No use of `sklearn.tree.DecisionTreeClassifier`
- Decision Tree implemented completely from scratch
- Random Forest implemented completely from scratch
- Evaluation metrics implemented manually wherever applicable

Only basic Python libraries are used for numerical computation, data manipulation, and visualization.

## Import libraries


In [ ]:
import numpy as np
import pandas as pd

# Used to efficiently count the frequency of values (e.g., for finding the most common class or mode).
from collections import Counter

# Plotting graphs and charts
import matplotlib.pyplot as plt

# Statistical data visualization
import seaborn as sns

## Data loading

In [ ]:
# Load the training and testing datasets
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

# Display the first five rows of each dataset
print("Training Dataset:")
display(train_df.head())

print("\nTesting Dataset:")
display(test_df.head())

# Display the number of rows and columns
print(f"Training Dataset Shape: {train_df.shape}")
print(f"Testing Dataset Shape: {test_df.shape}")

## Exploratory Data Analysis


### Dataset overview


In [ ]:
# Display information about the training dataset
train_df.info()

In [ ]:
test_df.info()

#### Observations

The training dataset contains 891 passenger records with 12 features, while the testing dataset contains 418 passenger records with 11 features (excluding the target variable, Survived). Both datasets include a combination of numerical and categorical variables. In the training dataset, missing values are present in the Age, Cabin, and Embarked features, while in the testing dataset, missing values occur in Age, Cabin, and Fare. The Cabin feature contains the largest number of missing values in both datasets, followed by Age. Only a small number of missing values are present in Embarked (training dataset) and Fare (testing dataset). The appropriate treatment for these missing values will be determined during the data preprocessing stage.

### Visual Analysis

#### Survival Rate by Passenger Class

In [ ]:
# Calculate the average survival rate for each passenger class
survival_by_class = (
    train_df.groupby("Pclass")["Survived"]
    .mean() * 100
)

# Create the bar chart
plt.figure(figsize=(6,4))

plt.bar(
    survival_by_class.index.astype(str),
    survival_by_class.values
)

# Graph labels
plt.xlabel("Passenger Class")
plt.ylabel("Survival Rate (%)")
plt.title("Survival Rate by Passenger Class")

plt.show()

Observation:
Higher-class passengers generally had higher survival rates.

#### Age Distribution of Survivors vs Non-Survivors

In [ ]:

plt.figure(figsize=(8,5))

# Age distribution of passengers who survived
plt.hist(
    train_df[train_df["Survived"] == 1]["Age"],
    bins=20,
    alpha=0.6,
    density=True,
    label="Survived"
)

# Age distribution of passengers who did not survive
plt.hist(
    train_df[train_df["Survived"] == 0]["Age"],
    bins=20,
    alpha=0.6,
    density=True,
    label="Did Not Survive"
)

# Graph labels
plt.xlabel("Age")
plt.ylabel("Density")
plt.title("Age Distribution of Survivors vs Non-Survivors")

plt.legend()

plt.show()

Observation:
The age distributions of survivors and non-survivors overlap considerably, indicating that age alone was not a strong predictor of survival. However, a relatively higher proportion of young children survived, suggesting that younger passengers generally had better chances of survival than adults.

#### Survival rate by age group

In [ ]:
# --------------------------------------------------
# Survival Rate by Age Group
# --------------------------------------------------

# Create age groups (5-year intervals)
age_bins = np.arange(0, 85, 5)

# Create labels for each age group
age_labels = [
    f"{age_bins[i]}-{age_bins[i+1]-1}"
    for i in range(len(age_bins)-1)
]
age_groups = pd.cut(
    train_df["Age"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

# Calculate survival rate for each age group
survival_by_age = (
    train_df
    .groupby(age_groups)["Survived"]
    .mean()
    * 100
)

# Remove empty age groups
survival_by_age = survival_by_age.dropna()

# --------------------------------------------------
# Plot
# --------------------------------------------------

plt.figure(figsize=(10,5))

plt.bar(
    survival_by_age.index.astype(str),
    survival_by_age.values
)

# Rotate labels for readability
plt.xticks(rotation=45)

# Graph labels
plt.xlabel("Age Group")
plt.ylabel("Survival Rate (%)")
plt.title("Survival Rate by Age Group")

# Display survival percentage above every bar
for i, value in enumerate(survival_by_age.values):

    plt.text(
        i,
        value + 1,
        f"{value:.1f}%",
        ha="center",
        fontsize=8
    )

plt.ylim(0, 100)

plt.tight_layout()

plt.show()

####Gender-Based Survival Analysis

In [ ]:
# Calculate the survival rate for each gender
survival_by_gender = (
    train_df.groupby("Sex")["Survived"]
    .mean() * 100
)

# Reorder so Male appears first
survival_by_gender = survival_by_gender.reindex(
    ["male", "female"]
)

plt.figure(figsize=(5,4))

plt.bar(
    survival_by_gender.index,
    survival_by_gender.values
)

plt.xlabel("Gender")
plt.ylabel("Survival Rate (%)")
plt.title("Gender-Based Survival Rate")

plt.ylim(0, 100)

plt.show()

Observation: Female passengers survived at a significantly higher rate than male passengers.

#### Fare Distribution Analysis

In [ ]:

plt.figure(figsize=(7,4))

# Plot the fare distribution
plt.hist(
    train_df["Fare"],
    bins=60
)

# Focus on the main fare range
plt.xlim(0, 150)

# Display x-axis labels every £10
plt.xticks(
    np.arange(0, 151, 10)
)

# Graph labels
plt.xlabel("Fare (£)")
plt.ylabel("Number of Passengers")
plt.title("Fare Distribution")

plt.show()

Observation:

The fare distribution is heavily right-skewed. Most passengers paid relatively low fares (below approximately £50), while only a small number purchased very expensive tickets. A few high-fare outliers exist, but limiting the x-axis to £150 improves visualization of the main distribution without affecting the overall trend.

### Survival Analysis

This section examines the distribution of the target variable (**Survived**) to understand how passengers are divided between the two outcome classes. This helps identify whether the dataset is balanced or imbalanced, providing important context for interpreting the model's performance and evaluation metrics.

In [ ]:
# Display the number of passengers who survived and did not survive
print(train_df["Survived"].value_counts())

In [ ]:
# Visualize the distribution of the target variable
plt.figure(figsize=(6, 4))
sns.countplot(data=train_df, x="Survived")

plt.title("Passenger Survival Distribution")
plt.xlabel("Survival Status")
plt.ylabel("Number of Passengers")

plt.show()

The dataset contains 549 passengers who did not survive and 342 passengers who survived, corresponding to approximately 61.6% and 38.4% of the dataset, respectively.

The target classes are slightly imbalanced, with a greater number of non-survivors than survivors. However, the imbalance is not severe, allowing the Random Forest classifier to learn meaningful patterns from both classes without the need for specialized techniques to handle class imbalance.

## Data Preprocessing

Data preprocessing prepares the dataset for machine learning by handling missing values, converting categorical variables into numerical representations, and selecting the features to be used for training the model. These steps ensure that the data is in a suitable format for the Random Forest classifier.

### Handling Missing Values

Missing values can reduce the quality of a machine learning model and, in some cases, prevent it from making predictions. Therefore, appropriate techniques are applied to handle missing values in both the training and testing datasets while preserving as much useful information as possible.

#### Cabin

The Cabin feature was the first missing-value problem I considered. Cabin information could potentially improve the model because the deck on which a passenger stayed may reflect factors such as cabin location, passenger class, and proximity to lifeboats, all of which could have influenced survival. However, more than 75% of the values were missing in both the training and testing datasets. I explored several ways to preserve this feature, including replacing missing values with the most common cabin and even using only the cabin deck (e.g., C85 → C) instead of the full cabin number. However, with so few known cabin values, none of these methods could estimate the missing data reliably.

Rather than introducing a large amount of artificial data, I decided to remove the Cabin feature from both datasets. Although this results in the loss of some potentially useful information, it produces a cleaner and more reliable dataset for training the model and generating predictions.

In [ ]:
# Remove the Cabin column due to the large number of missing values
train_df.drop(columns=["Cabin"], inplace=True)
test_df.drop(columns=["Cabin"], inplace=True)

print("Training Dataset:")
display(train_df.head())

print("\nTesting Dataset:")
display(test_df.head())

#### Embarked

The Embarked feature indicates the port from which each passenger boarded the Titanic. Only two passengers in the training dataset have missing embarkation values, while the testing dataset contains no missing values for this feature. Removing the feature would therefore unnecessarily discard useful information.

Instead, I decided to estimate the missing values using passengers with similar characteristics. For example, if a passenger belongs to Pclass = 2, Sex = Female, SibSp = 1, and Parch = 0, I first look for passengers with the same characteristics. If at least ten matching passengers are found, the missing value is replaced with the most common embarkation port (mode) within that group. I chose a minimum of ten passengers to ensure that the estimate was based on a sufficiently representative group rather than a very small sample, which could make the imputed value unreliable. If fewer than ten passengers are available, the matching criteria are gradually relaxed until a sufficiently large group is obtained.

In the event of a tie between multiple modes, the same tied values are compared within the next less specific group in the hierarchy. This process continues until a single mode is obtained. If a tie still remains after considering the entire dataset, the first value in the final set of modes is selected.

This approach estimates the missing values using the most similar passengers instead of assigning the same embarkation port to every missing record, while preserving as much passenger-specific information as possible.

#### Age

The Age feature is one of the most important features in the dataset and contains missing values in both the training and testing datasets. Therefore, removing the feature or deleting passengers with missing ages would result in the loss of valuable information.

The same hierarchical imputation strategy described for the Embarked feature is used here. The only difference is that, since Age is a numerical feature, the missing values are replaced with the median age of the most similar passenger group instead of the mode. The Embarked feature, which has already been imputed, is also included as one of the grouping features to improve the similarity between passengers.

For example, if the most similar group contains the ages 22, 24, 25, 26, 29, 30, 31, 32, 33, 35, the missing age would be replaced with the median of that group.

#### Fare
The Fare feature contains only one missing value, which occurs in the testing dataset. Although the number of missing values is very small, leaving it untreated could prevent the trained model from making predictions. Rather than replacing it with the overall median fare, I applied the same hierarchical grouping strategy used for the other features to maintain consistency throughout the preprocessing pipeline. Since Fare is a numerical feature, the missing value is replaced with the median fare of the most similar passenger group.

#### Hierarchical Group Search Function

Since the Embarked, Age, and Fare imputation strategies all follow the same hierarchical grouping procedure, the common search logic has been implemented as a reusable function. The function identifies the most similar passenger group by progressively relaxing the grouping criteria until a sufficiently large group is found. In addition to returning the final matching group, it also stores every intermediate group encountered during the search. This allows the same hierarchy to be reused later without performing the search again.

In [ ]:
def find_matching_group(
    df,
    index,
    features,
    target,
    min_group_size=10
):
    """
    Find the most specific group with enough non-missing values
    for hierarchical imputation.
    """

    # Store every group created during the search
    hierarchy = []

    # Start with all features and progressively remove one feature
    # until a sufficiently large group is found
    for i in range(
        len(features),
        -1,
        -1
    ):

        # Current combination of features to match
        current_features = features[:i]

        # If no features remain, use the entire dataset
        if len(current_features) == 0:

            group = df[
                df[target].notna()
            ]

        else:

            # Start with the complete dataset
            group = df.copy()

            # Keep only rows matching the current passenger
            # for every selected feature
            for feature in current_features:

                group = group[
                    group[feature]
                    ==
                    df.loc[index, feature]
                ]

            # Remove rows where the target value is missing
            group = group[
                group[target].notna()
            ]

        # Save the current group for visualization
        hierarchy.append(group)

        # Stop if the group is large enough
        if len(group) >= min_group_size:

            return group, hierarchy

    # If no group reaches the minimum size,
    # return the broadest available group
    return hierarchy[-1], hierarchy

#### Mode Resolution function
A second helper function is used when imputing categorical features such as Embarked. If the selected group contains a single mode, that value is used directly. If multiple modes are present, the function compares only the tied values within the next less specific group in the stored hierarchy. This process continues until a single mode is obtained. If the tie still remains after considering the entire dataset, the first value in the final set of modes is selected. By reusing the stored hierarchy from the search function, this tie-breaking process avoids repeating the grouping calculations while producing a more reliable estimate

In [ ]:
def resolve_mode(hierarchy, target):

    modes = hierarchy[0][target].mode()

    # Only one mode
    if len(modes) == 1:
        return modes.iloc[0]

    # Walk through progressively larger groups
    for group in hierarchy[1:]:

        counts = group[target].value_counts().reindex(modes, fill_value=0)

        if counts.max() > counts.min():
            return counts.idxmax()

    # Still tied in the entire dataset
    return modes.iloc[0]

#### Embarked Implementation

In [ ]:
# --------------------------------------------------
# Impute Missing 'Embarked' Values
# --------------------------------------------------

# Iterate through every passenger with a missing Embarked value
for index in train_df[
    train_df["Embarked"].isnull()
].index:

    # Find the most specific matching group
    group, hierarchy = find_matching_group(
        train_df,
        index,
        ["Pclass", "Sex", "SibSp", "Parch"],
        "Embarked"
    )

    # Impute using the hierarchical mode
    train_df.loc[index, "Embarked"] = resolve_mode(
        hierarchy,
        "Embarked"
    )

# --------------------------------------------------
# Verify Imputation
# --------------------------------------------------

print("Missing 'Embarked' values after imputation:")

print(
    f"Training Dataset: "
    f"{train_df['Embarked'].isnull().sum()}"
)

print(
    f"Testing Dataset: "
    f"{test_df['Embarked'].isnull().sum()}"
)

#### Age Implementation

In [ ]:
# Impute missing Age values in the training dataset
for index in train_df[train_df["Age"].isnull()].index:

    group, _ = find_matching_group(
        train_df,
        index,
        ["Pclass", "Sex", "SibSp", "Parch", "Embarked"],
        "Age"
    )

    train_df.loc[index, "Age"] = group["Age"].median()


# Impute missing Age values in the testing dataset
for index in test_df[test_df["Age"].isnull()].index:

    group, _ = find_matching_group(
        test_df,
        index,
        ["Pclass", "Sex", "SibSp", "Parch", "Embarked"],
        "Age"
    )

    test_df.loc[index, "Age"] = group["Age"].median()


# Verify that no missing Age values remain
print("Missing 'Age' values after imputation:")
print(f"Training Dataset: {train_df['Age'].isnull().sum()}")
print(f"Testing Dataset: {test_df['Age'].isnull().sum()}")

#### Fare implementation

In [ ]:
# Impute missing Fare values in the testing dataset
for index in test_df[test_df["Fare"].isnull()].index:

    group, _ = find_matching_group(
        test_df,
        index,
        ["Pclass", "Sex", "SibSp", "Parch", "Embarked"],
        "Fare"
    )

    test_df.loc[index, "Fare"] = group["Fare"].median()


# Verify that no missing Fare values remain
print("Missing 'Fare' values after imputation:")
print(f"Training Dataset: {train_df['Fare'].isnull().sum()}")
print(f"Testing Dataset: {test_df['Fare'].isnull().sum()}")

In [ ]:
print(test_df.isnull().sum())

In [ ]:
print(train_df.isnull().sum())

### Feature Selection

Feature selection determines which features should be retained for further processing. Instead of removing features simply because they are not immediately useful, I first considered whether they contained information that could be transformed into more meaningful features during the feature engineering stage.

| Feature | Decision | Justification |
|:--------|:--------:|:--------------|
| **PassengerId** |  Remove | Unique identifier with no predictive value. |
| **Pclass** |  Keep | Indicates passenger class, which may influence survival. |
| **Name** |  Keep | Contains passenger titles that can be extracted during feature engineering. |
| **Sex** |  Keep | One of the strongest predictors of survival. |
| **Age** |  Keep | May influence rescue priority and survival. |
| **SibSp** |  Keep | Used to derive the **FamilySize** feature during feature engineering. |
| **Parch** |  Keep | Used to derive the **FamilySize** feature during feature engineering. |
| **Ticket** |  Remove | Ticket values are highly inconsistent and difficult to transform into meaningful features for this project. |
| **Fare** |  Keep | Reflects ticket price and socioeconomic status. |
| **Embarked** |  Keep | Boarding port may capture demographic differences. |

In [ ]:
# Remove features that will not be used for model training
train_df.drop(columns=["PassengerId", "Ticket"], inplace=True)
test_df.drop(columns=["PassengerId", "Ticket"], inplace=True)

# Display the updated datasets
print("Training Dataset:")
display(train_df.head())

print("\nTesting Dataset:")
display(test_df.head())

### Feature Engineering

Feature engineering transforms existing features into representations that are more meaningful for the model. Rather than relying only on the information directly available in the dataset, new features can be created by combining or extracting useful information from existing ones.

For this project, two new features are created:

- **Title**, extracted from the **Name** feature, to capture information such as social status and age group.
- **FamilySize**, derived from **SibSp** and **Parch**, to represent the total number of family members travelling together.

After these new features are created, the original features used to construct them are removed to avoid retaining redundant information in the dataset.

#### Title

The **Name** feature was retained because it contains each passenger's **title**, which can be extracted as a separate feature.

 **Why extract the Title if the dataset already has Age and Sex?**

Some titles provide additional context that may help distinguish passengers with otherwise similar characteristics.

For example:

- **Mr** – Adult male
- **Master** – Young boy
- **Mrs** – Married female
- **Miss** – Unmarried female
- **Dr, Rev, Col, Lady, Sir** – Profession, military rank, or social status

However, many of these uncommon titles appear only once or twice in the dataset, making it difficult for the model to learn reliable patterns from them individually.

To reduce sparsity while preserving useful information, the titles were grouped into five categories:

- **Mr**
- **Mrs**
- **Miss**
- **Master**
- **Rare** *(all remaining uncommon titles)*

Finally, the grouped titles are **one-hot encoded**, allowing the Decision Tree to evaluate each category independently without introducing an artificial numerical ordering.

Once the titles have been extracted, the original **Name** feature is removed because it no longer provides any additional useful information.

In [ ]:
# Extract the Title from the Name feature
train_df["Title"] = train_df["Name"].str.extract(r",\s*([^\.]+)\.")
test_df["Title"] = test_df["Name"].str.extract(r",\s*([^\.]+)\.")

# Group uncommon titles into a single "Rare" category
title_mapping = {
    "Mr": "Mr",
    "Mrs": "Mrs",
    "Miss": "Miss",
    "Master": "Master",
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
}

train_df["Title"] = train_df["Title"].apply(
    lambda x: title_mapping[x] if x in title_mapping else "Rare"
)

test_df["Title"] = test_df["Title"].apply(
    lambda x: title_mapping[x] if x in title_mapping else "Rare"
)

# Remove the original Name feature
train_df.drop(columns=["Name"], inplace=True)
test_df.drop(columns=["Name"], inplace=True)

# Display the updated datasets
print("Training Dataset:")
display(train_df.head())

print("\nTesting Dataset:")
display(test_df.head())

#### FamilySize

The **SibSp** and **Parch** features were retained during feature selection because they can be combined to create a new feature called **FamilySize**.

A natural question is:

**Why create FamilySize instead of using SibSp and Parch separately?**

The answer is that the **total number of family members travelling with a passenger** is often more informative than distinguishing whether those family members were siblings, spouses, parents, or children. For example:

- **SibSp = 1, Parch = 2** → **FamilySize = 4**
- **SibSp = 2, Parch = 1** → **FamilySize = 4**

Although these passengers have different family compositions, they are both travelling with **three relatives**. The overall family size is therefore likely to be a more meaningful indicator than the individual counts.

Therefore, the **FamilySize** feature is created by combining **SibSp** and **Parch**. Once this new feature has been created, the original **SibSp** and **Parch** features are removed to avoid retaining redundant information in the dataset.

In [ ]:
# Create the FamilySize feature
train_df["FamilySize"] = train_df["SibSp"] + train_df["Parch"] + 1
test_df["FamilySize"] = test_df["SibSp"] + test_df["Parch"] + 1

# Remove the original SibSp and Parch features
train_df.drop(columns=["SibSp", "Parch"], inplace=True)
test_df.drop(columns=["SibSp", "Parch"], inplace=True)

# Display the updated datasets
print("Training Dataset:")
display(train_df.head())

print("\nTesting Dataset:")
display(test_df.head())

Continuous Variables: Although the task suggested binning Age and Fare, these features were intentionally retained as continuous variables. Decision Trees and Random Forests naturally determine optimal split thresholds during training, allowing the model to make finer-grained decisions than fixed bins while preserving all available information.

### Encoding Categorical Features

The **Sex**, **Embarked**, and **Title** features are categorical and must therefore be converted into numerical values before training the Random Forest model.

A natural question is:

**Why use different encoding techniques?**

- **Sex** contains only two categories (*Male* and *Female*), so **label encoding** is sufficient.
- **Embarked** and **Title** contain multiple categories with **no natural ordering**. Using label encoding would introduce an artificial numerical relationship (e.g., `C < Q < S`), which could restrict the Decision Tree's possible splits. Therefore, **one-hot encoding** was used instead.

The final encoding strategy is:

- **Sex** → Label Encoding (`Male = 0`, `Female = 1`)
- **Embarked** → One-Hot Encoding (`Embarked_C`, `Embarked_Q`, `Embarked_S`)
- **Title** → One-Hot Encoding (`Mr`, `Mrs`, `Miss`, `Master`, `Rare`)

This ensures that each category is treated independently, allowing the Decision Tree to evaluate every category without introducing unnecessary bias.

In [ ]:
# Label encode the Sex feature
sex_mapping = {
    "male": 0,
    "female": 1
}

train_df["Sex"] = train_df["Sex"].map(sex_mapping)
test_df["Sex"] = test_df["Sex"].map(sex_mapping)


# One-hot encode the Embarked feature
train_df = pd.get_dummies(
    train_df,
    columns=["Embarked"],
    prefix="Embarked",
    dtype=int
)

test_df = pd.get_dummies(
    test_df,
    columns=["Embarked"],
    prefix="Embarked",
    dtype=int
)


# One-hot encode the Title feature
train_df = pd.get_dummies(
    train_df,
    columns=["Title"],
    prefix="Title",
    dtype=int
)

test_df = pd.get_dummies(
    test_df,
    columns=["Title"],
    prefix="Title",
    dtype=int
)


# Ensure both datasets contain the same one-hot encoded columns
train_df, test_df = train_df.align(
    test_df,
    join="left",
    axis=1,
    fill_value=0
)


display(
    train_df.filter(
        regex="^(Sex|Embarked_|Title_)"
    ).head()
)

display(
    test_df.filter(
        regex="^(Sex|Embarked_|Title_)"
    ).head()
)

### Splitting Features and Target Variable

Machine learning models require the input features and the target variable to be separated before training.

The **Survived** feature is the target variable because it represents the value that the model is expected to predict. All remaining features are treated as the input features used to make that prediction.

Since the testing dataset does not contain the **Survived** feature, only the input features are extracted from it.

In [ ]:
# Split the training dataset into features (X) and target (y)
X_train = train_df.drop(columns=["Survived"])
y_train = train_df["Survived"]

# Extract the input features from the testing dataset
X_test = test_df.copy()

# Display the resulting datasets
print("Training Features (X_train):")
display(X_train.head())

print("\nTraining Target (y_train):")
display(y_train.head())

print("\nTesting Features (X_test):")
display(X_test.head())

## Converting DataFrames to NumPy Arrays

The Decision Tree and Random Forest developed in this project are implemented entirely from scratch using NumPy. These implementations perform array indexing operations such as `X[:, feature]`, which are supported by NumPy arrays but not by Pandas DataFrames.

Therefore, once all preprocessing and the train-test split have been completed, the datasets are converted to NumPy arrays before training the models.

In [ ]:
# Convert the datasets to NumPy arrays

X_train = X_train.to_numpy()
y_train = y_train.to_numpy()

X_test = X_test.to_numpy()

## Decision Tree Implementation

### Default Hyperparameters

To keep the implementation focused, the Random Forest is initially built using the following **default hyperparameters**:

- **Number of Trees:** 100
- **Maximum Tree Depth:** 5
- **Minimum Samples for Split:** 2

These values are **temporary** and are only used while implementing and verifying the algorithm. Once the implementation is complete, these hyperparameters will be tuned to determine the combination that produces the best validation performance.

### Node Structure

A decision tree is composed of **nodes**, where each node represents either a decision or a prediction.

- An **internal node** stores the feature and threshold used to split the data into two groups.
- A **leaf node** stores the final prediction and marks the end of a branch.

A leaf node is created whenever the tree stops splitting. This can happen because:

- all samples in the node belong to the same class (pure node),
- the maximum tree depth has been reached,
- the node contains fewer samples than the minimum samples required for splitting, or
- no valid split can further reduce the impurity.

Each node therefore stores the information required to either continue traversing the tree or return a prediction. The tree itself is formed by linking nodes together through its left and right child nodes.

In [ ]:
class Node:
    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        value=None,
        information_gain=0,
        num_samples=0
    ):

        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

        self.information_gain = information_gain
        self.num_samples = num_samples

### Gini Impurity

To determine the best split at each node, the decision tree measures how mixed the samples are using **Gini Impurity**.

The Gini Impurity of a node is calculated as

$$
G = 1 - \sum_{i=1}^{c} p_i^2
$$

where \(p_i\) is the proportion of samples belonging to class \(i\).

For this binary classification problem:

- **Gini = 0** → Pure node (all samples belong to one class)
- **Gini = 0.5** → Maximum impurity (both classes are equally mixed)

When evaluating a split, the tree computes the **weighted Gini Impurity** of the two child nodes:

$$
G_{\text{weighted}}
=
\frac{N_L}{N}G_L
+
\frac{N_R}{N}G_R
$$

where \(N_L\) and \(N_R\) are the number of samples in the left and right child nodes, \(N\) is the total number of samples, and \(G_L\) and \(G_R\) are the Gini Impurities of the child nodes.

The split with the **lowest weighted Gini Impurity** is selected, since it produces the purest child nodes.


In [ ]:
def gini_impurity(self,y):
    """
    Calculate the Gini Impurity of a node.

    Parameters:
        y (numpy.ndarray): Array containing the class labels.

    Returns:
        float: Gini Impurity of the node.
    """

    # Empty nodes are perfectly pure
    if len(y) == 0:
        return 0.0

    # Count the occurrences of each class
    _, counts = np.unique(y, return_counts=True)

    # Compute the probability of each class
    probabilities = counts / len(y)

    # Apply the Gini Impurity formula
    gini = 1 - np.sum(probabilities ** 2)

    return gini

In [ ]:
def weighted_gini(
    self,
    left_y,
    right_y
):
    """
    Calculate the weighted Gini Impurity after a split.
    """

    # Total number of samples before the split
    total = len(left_y) + len(right_y)

    # Proportion of samples in the left child
    left_weight = len(left_y) / total

    # Proportion of samples in the right child
    right_weight = len(right_y) / total

    # Weighted average of the two child impurities
    return (
        left_weight * self.gini_impurity(left_y)
        +
        right_weight * self.gini_impurity(right_y)
    )

### Information Gain

While Gini Impurity measures how mixed a node is, **Information Gain** measures how much a split improves the purity of the data.

It is calculated as the difference between the Gini Impurity of the parent node and the weighted Gini Impurity of its child nodes:

$$
\text{Information Gain}
=
G_{\text{parent}}
-
G_{\text{weighted}}
$$

A larger Information Gain indicates that the split produces purer child nodes and is therefore a better split. During tree construction, every possible split is evaluated, and the split with the **highest Information Gain** is selected.

In [ ]:
def information_gain(self,parent_y, left_y, right_y):
    """
    Calculate the Information Gain of a split.

    Parameters:
        parent_y (numpy.ndarray): Labels in the parent node.
        left_y (numpy.ndarray): Labels in the left child node.
        right_y (numpy.ndarray): Labels in the right child node.

    Returns:
        float: Information Gain of the split.
    """

    # Gini Impurity of the parent node
    parent_gini = self.gini_impurity(parent_y)

    # Weighted Gini Impurity of the child nodes
    children_gini = self.weighted_gini(left_y, right_y)

    # Information Gain
    return parent_gini - children_gini

### Finding the Best Split

If every decision tree always considered **all features**, many trees would learn very similar splitting patterns. This would reduce the diversity of the forest and increase the risk of overfitting.

To encourage each tree to learn differently, the Random Forest randomly selects **√n** features at every node, where **n** is the total number of input features.

 **Example:**  
 If the dataset has **7 features**, then  
 √7 ≈ **2.65**, so only **2 features** are randomly selected.

For the selected features:

1. Evaluate every possible split.
2. Calculate the **Information Gain** of each split.
3. Choose the split with the **highest Information Gain**.

By exposing different trees to different subsets of features, the Random Forest produces a more diverse collection of decision trees, improving its ability to generalize to unseen data.

#### For spliting the dataset

In [ ]:
def split_dataset(
    self,
    X,
    y,
    feature_index,
    threshold
):
    """
    Split the dataset into left and right child datasets.
    """

    left_indices = np.where(
        X[:, feature_index] <= threshold
    )[0]

    right_indices = np.where(
        X[:, feature_index] > threshold
    )[0]

    X_left = X[left_indices]
    X_right = X[right_indices]

    y_left = y[left_indices]
    y_right = y[right_indices]

    return (
        X_left,
        X_right,
        y_left,
        y_right
    )

#### For finding the best split

In [ ]:
def best_split(self, X, y):
    """
    Find the best feature and threshold for splitting.

    Parameters:
        X : Feature matrix
        y : Target labels

    Returns:
        best_feature, best_threshold
    """

    num_samples, num_features = X.shape

    # ---------------------------------------------------------
    # Determine how many random features to consider
    # ---------------------------------------------------------
    if self.max_features == "sqrt":
        max_features = max(
            1,
            int(np.sqrt(num_features))
        )

    elif self.max_features is None:
        # Use all features (standard Decision Tree)
        max_features = num_features

    else:
        # Use the specified number of features
        max_features = min(
            self.max_features,
            num_features
        )

    # ---------------------------------------------------------
    # Randomly select feature indices
    # ---------------------------------------------------------
    feature_indices = np.random.choice(
        num_features,
        max_features,
        replace=False
    )

    best_gain = -1
    best_feature = None
    best_threshold = None

    # ---------------------------------------------------------
    # Evaluate every selected feature
    # ---------------------------------------------------------
    for feature in feature_indices:

        # Candidate thresholds
        thresholds = np.unique(
            X[:, feature]
        )

        for threshold in thresholds:

            # Split the samples
            left_idx = np.where(
                X[:, feature] <= threshold
            )[0]

            right_idx = np.where(
                X[:, feature] > threshold
            )[0]

            # Ignore invalid splits
            if (
                len(left_idx) == 0
                or
                len(right_idx) == 0
            ):
                continue

            # Calculate Information Gain
            gain = self.information_gain(
                y,
                y[left_idx],
                y[right_idx]
            )

            # Update the best split
            if gain > best_gain:

                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold, best_gain

### Selecting the Majority Class (`most_common_label()`)

Not every leaf node is created because it is **pure**.

A leaf node is also created when:

- The **maximum depth** is reached.
- The node contains fewer than the **minimum samples required to split**.
- No valid split can further reduce the Gini Impurity.

In these situations, the node may still contain a mixture of classes. The `most_common_label()` function assigns the class that appears **most frequently** within that node as its final prediction.

For example:

```text
Current Node

Survived = 1   → 7 passengers
Survived = 0   → 3 passengers
```

The node predicts:

```text
Survived = 1
```

This ensures that every leaf node is able to produce a prediction, even when it is not completely pure.

In [ ]:
def most_common_label(self, y):
    """
    Return the most common class label in a node.

    Parameters:
        y : Target labels

    Returns:
        The majority class label.
    """

    # Count the occurrences of each class
    class_counts = Counter(y)

    # Return the class with the highest frequency
    return class_counts.most_common(1)[0][0]

### Recursive Tree Construction (`build_tree()`)

The `build_tree()` function is the **core of the Decision Tree algorithm**. Starting from the root node, it repeatedly finds the best split and creates child nodes until a stopping condition is reached.

At every node, the following steps are performed:

1. Check whether a **leaf node** should be created.
2. Find the **best feature** and **best threshold** for splitting.
3. Divide the dataset into **left** and **right** child datasets.
4. Recursively repeat the same process for each child.

```text
                Root
                  │
          Best Split Found
             /          \
        Left Node     Right Node
          /   \         /    \
        ...   ...     ...    ...
```

The recursion stops when:

- The node becomes **pure**.
- The **maximum depth** is reached.
- The node contains fewer than the **minimum samples required to split**.
- No valid split can further reduce the **Gini Impurity**.

The function then returns the completed subtree, eventually constructing the entire Decision Tree from the root to the leaf nodes.

In [ ]:
def build_tree(
    self,
    X,
    y,
    depth
):
    """
    Recursively construct the Decision Tree.

    Parameters:
        X : Feature matrix
        y : Target labels
        depth : Current depth of the tree

    Returns:
        Root node of the constructed subtree.
    """

    # Number of samples at the current node
    num_samples = len(y)

    # -------------------------
    # Stopping Condition 1
    # Node is pure
    # -------------------------
    if len(np.unique(y)) == 1:
        return Node(value=y[0])

    # -------------------------
    # Stopping Condition 2
    # Maximum depth reached
    # -------------------------
    if (
        self.max_depth is not None
        and depth >= self.max_depth
    ):
        return Node(
            value=self.most_common_label(y)
        )

    # -------------------------
    # Stopping Condition 3
    # Too few samples to split
    # -------------------------
    if num_samples < self.min_samples_split:
        return Node(
            value=self.most_common_label(y)
        )

     # Find the best split
    best_feature, best_threshold, best_gain = self.best_split(
        X,
        y
    )

    # No valid split found
    if best_feature is None:
        return Node(
            value=self.most_common_label(y)
        )

    # Split the dataset
    (
        X_left,
        X_right,
        y_left,
        y_right
    ) = self.split_dataset(
        X,
        y,
        best_feature,
        best_threshold
    )

    # Build the left subtree
    left_child = self.build_tree(
        X_left,
        y_left,
        depth + 1
    )

    # Build the right subtree
    right_child = self.build_tree(
        X_right,
        y_right,
        depth + 1
    )

    # Return the current decision node
    return Node(
    feature=best_feature,
    threshold=best_threshold,
    left=left_child,
    right=right_child,
    information_gain=best_gain,
    num_samples=num_samples
)

### Model Training (`fit()`)

The `fit()` method is responsible for training the Decision Tree.

Rather than constructing the tree directly, it simply starts the recursive tree-building process from the root node. The completed tree is then stored as the root of the `DecisionTree` object, allowing it to be used later for prediction.

In [ ]:
def fit(self, X, y):
    """
    Train the Decision Tree using the training dataset.

    Parameters:
        X : Feature matrix
        y : Target labels
    """

    # Start building the tree from the root node
    self.root = self.build_tree(
        X,
        y,
        depth=0
    )

### Predicting a Single Sample (`predict_sample()`)

Once the Decision Tree has been trained, it can classify a new passenger by **traversing the tree** from the root node.

At each decision node:

- The passenger's feature value is compared with the node's threshold.
- If the condition is **True**, the traversal moves to the **left child**.
- Otherwise, it moves to the **right child**.

This process continues until a **leaf node** is reached. The class stored in that leaf node becomes the prediction for the passenger.

For example:

```text
           Sex ≤ 0
          /       \
       Yes         No
      Age ≤ 12    Survived = 1
      /     \
    Yes     No
Survived=1 Survived=0
```

For a **male passenger aged 10**:

- `Sex ≤ 0` → Yes
- `Age ≤ 12` → Yes
- **Prediction → Survived = 1**

The `predict_sample()` function performs this traversal for a **single passenger**, while the `predict()` function applies the same process to every passenger in the dataset.

In [ ]:
def predict_sample(self, x):
    """
    Predict the class label for a single sample.

    Parameters:
        x : A single sample (feature vector)

    Returns:
        Predicted class label.
    """

    # Start from the root node
    node = self.root

    # Traverse the tree until a leaf node is reached
    while node.value is None:

        # Move to the left child if the condition is satisfied
        if x[node.feature] <= node.threshold:
            node = node.left

        # Otherwise move to the right child
        else:
            node = node.right

    # Return the prediction stored in the leaf node
    return node.value

### Predicting Multiple Samples (`predict()`)

The `predict_sample()` function classifies **one passenger at a time**.

To classify an entire dataset, the `predict()` function simply repeats this process for **every passenger** and stores the predictions in a single array.

```text
Dataset
   │
   ├── Passenger 1 → predict_sample() → Prediction
   ├── Passenger 2 → predict_sample() → Prediction
   ├── Passenger 3 → predict_sample() → Prediction
   └── ...

           ↓

Predictions = [1, 0, 0, 1, ...]
```

This function allows the trained Decision Tree to generate predictions for the complete testing or validation dataset in a single call.

In [ ]:
def predict(self, X):
    """
    Predict the class labels for multiple samples.

    Parameters:
        X : Feature matrix

    Returns:
        NumPy array containing the predicted class labels.
    """

    # Predict the class label for every sample
    predictions = [
        self.predict_sample(sample)
        for sample in X
    ]

    # Return all predictions as a NumPy array
    return np.array(predictions)

### Decision Tree Class

The final `DecisionTree` class combines all the previously implemented functions into a single model.

It stores the **hyperparameters**, **constructs the tree during training**, and **uses the trained tree to make predictions**

The `DecisionTree` class is now a complete machine learning model that can:

- Train on a dataset using `fit()`.
- Build the tree recursively.
- Predict the class of a **single passenger** using `predict_sample()`.
- Predict the classes of an **entire dataset** using `predict()`.

This Decision Tree will serve as the **base learner** for the Random Forest model, where multiple independent trees are trained and their predictions are combined to produce the final result.

In [ ]:
class DecisionTree:

    def __init__(
    self,
    max_depth=None,
    min_samples_split=2,
    max_features="sqrt"
):
        # Maximum depth allowed for the tree
        self.max_depth = max_depth

        # Minimum number of samples required to split a node
        self.min_samples_split = min_samples_split

        # Number of randomly selected features at each split
        self.max_features = max_features

        # Root node of the trained Decision Tree
        self.root = None


    # =========================================================
    # Impurity Measures
    # =========================================================

    gini_impurity = gini_impurity
    weighted_gini = weighted_gini


    # =========================================================
    # Split Evaluation
    # =========================================================

    information_gain = information_gain
    split_dataset = split_dataset
    best_split = best_split


    # =========================================================
    # Tree Construction
    # =========================================================

    most_common_label = most_common_label
    build_tree = build_tree
    fit = fit


    # =========================================================
    # Prediction
    # =========================================================

    predict_sample = predict_sample
    predict = predict

### Testing the decision tree

In [ ]:
tree = DecisionTree(
    max_depth=5,
    min_samples_split=2
)

print(tree)

In [ ]:
import numpy as np

X = np.array([
    [1],
    [2],
    [3],
    [8],
    [9],
    [10]
])

y = np.array([
    0,
    0,
    0,
    1,
    1,
    1
])

tree.fit(X, y)

In [ ]:
predictions = tree.predict(X)

print("Predictions :", predictions)
print("Actual      :", y)

In [ ]:
print("Root Feature   :", tree.root.feature)
print("Root Threshold :", tree.root.threshold)

In [ ]:
print("Left Child :", tree.root.left.value)
print("Right Child:", tree.root.right.value)

In [ ]:
tree = DecisionTree(
    max_depth=10,
    min_samples_split=2
)

tree.fit(X_train, y_train)

predictions = tree.predict(X_train)

accuracy = np.mean(predictions == y_train)

print(f"Training Accuracy: {accuracy:.4f}")

In [ ]:
print(tree.root.feature)
print(tree.root.threshold)

In [ ]:
print(tree.root.left)
print(tree.root.right)

## Random Forest Implementation

### Bootstrap Sampling

Bootstrap sampling is a resampling technique used to generate multiple training datasets from a single original dataset. Each bootstrap sample is created by randomly selecting training examples **with replacement**, meaning that the same sample may be selected multiple times while others may not be selected at all.

Since each Decision Tree is trained on a different bootstrap sample, every tree learns from a slightly different version of the training data. This increases the diversity of the trees and reduces the overall variance of the Random Forest, resulting in better generalisation on unseen data.

For a training dataset containing **N** samples, each bootstrap sample also contains **N** samples.

In [ ]:
def bootstrap_sample(self, X, y):
    """
    Generate a bootstrap sample from the training dataset.

    Parameters:
        X : Feature matrix
        y : Target labels

    Returns:
        X_sample : Bootstrap feature matrix
        y_sample : Bootstrap target labels
    """

    # Number of training samples
    n_samples = X.shape[0]

    # Randomly sample indices with replacement
    indices = np.random.choice(
        n_samples,
        size=n_samples,
        replace=True
    )

    # Create the bootstrap sample
    X_sample = X[indices]
    y_sample = y[indices]

    return X_sample, y_sample

### Random Feature Selection

One of the key ideas behind a Random Forest is that each Decision Tree considers only a random subset of the available features when searching for the best split at every intermediate node. This introduces additional randomness into the model, reducing the correlation between individual trees and improving the overall performance of the ensemble.

This functionality has already been implemented in the `DecisionTree.best_split()` method. At every node, the algorithm randomly selects **√n** features (where *n* is the total number of input features) and evaluates only those features when determining the optimal split. Since `best_split()` is called recursively during tree construction, a new random subset of features is selected independently for every intermediate node of every Decision Tree.

### Training Multiple Trees

After generating bootstrap samples, the Random Forest trains multiple Decision Trees independently. Each tree is trained on a different bootstrap sample while using random feature selection during split evaluation.

The training process is repeated for the specified number of trees (`n_estimators`). Once trained, every Decision Tree is stored in the Random Forest and later contributes to the final prediction through majority voting.

Since each tree is trained on a different subset of the data and considers different subsets of features at each split, the resulting trees are diverse. This diversity is the primary reason why a Random Forest generally performs better than a single Decision Tree.

In [ ]:
def fit(self, X, y):
    """
    Train multiple Decision Trees using bootstrap sampling.

    Parameters:
        X : Feature matrix
        y : Target labels
    """

    # Remove any previously trained trees
    self.trees = []

    # Train n_estimators Decision Trees
    for _ in range(self.n_estimators):

        # Create a bootstrap sample
        X_sample, y_sample = self.bootstrap_sample(
            X,
            y
        )

        # Create a new Decision Tree
        tree = DecisionTree(
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            max_features=self.max_features
        )

        # Train the Decision Tree
        tree.fit(
            X_sample,
            y_sample
        )

        # Store the trained tree
        self.trees.append(tree)

### Majority Voting

Once all Decision Trees have been trained, the Random Forest combines their predictions using **majority voting**.

Each Decision Tree independently predicts the class label for every sample. The Random Forest then counts the votes from all trees and selects the class that receives the highest number of votes as the final prediction.

Since every tree contributes equally to the final decision, incorrect predictions made by individual trees are often outweighed by the majority of correctly predicting trees. This collective decision-making process is one of the main reasons why Random Forests generally achieve better predictive performance than a single Decision Tree.

In [ ]:
def predict(self, X):
    """
    Predict class labels using majority voting.

    Parameters:
        X : Feature matrix

    Returns:
        NumPy array containing the predicted class labels.
    """

    # Collect predictions from every Decision Tree
    tree_predictions = np.array([
        tree.predict(X)
        for tree in self.trees
    ])

    # Majority vote for each sample
    predictions = []

    for sample_predictions in tree_predictions.T:

        values, counts = np.unique(
            sample_predictions,
            return_counts=True
        )

        predictions.append(
            values[np.argmax(counts)]
        )

    return np.array(predictions)

### Prediction Probabilities

In addition to predicting the final class label, a Random Forest can also estimate the probability of each class.

Instead of selecting only the majority vote, the algorithm counts how many Decision Trees predict each class and converts these counts into probabilities. The probability of a class is simply the proportion of trees that voted for that class.

For example, if a Random Forest contains 100 Decision Trees and 78 trees predict **Survived** while 22 predict **Did Not Survive**, the predicted probability would be:

- Survived: 78%
- Did Not Survive: 22%

These probabilities provide an indication of the model's confidence in its prediction and can be useful for evaluating prediction certainty.

In [ ]:
def predict_proba(self, X):
    """
    Predict class probabilities.

    Parameters:
        X : Feature matrix

    Returns:
        NumPy array containing the probability of each class.
    """

    # Collect predictions from every Decision Tree
    tree_predictions = np.array([
        tree.predict(X)
        for tree in self.trees
    ])

    probabilities = []

    # Calculate probabilities for every sample
    for sample_predictions in tree_predictions.T:

        values, counts = np.unique(
            sample_predictions,
            return_counts=True
        )

        sample_probabilities = np.zeros(2)

        sample_probabilities[values] = (
            counts / self.n_estimators
        )

        probabilities.append(
            sample_probabilities
        )

    return np.array(probabilities)

### Random Forest Class

The `RandomForest` class combines all previously implemented components into a single machine learning model.

During training, the class repeatedly generates bootstrap samples, trains multiple independent Decision Trees, and stores the trained trees. During prediction, the class combines the predictions from all Decision Trees using majority voting. It can also estimate prediction probabilities by calculating the proportion of trees that vote for each class.

This modular design makes the implementation easier to understand, maintain, and extend while closely following the standard Random Forest algorithm.

In [ ]:
class RandomForest:

    def __init__(
        self,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        max_features="sqrt"
    ):
        # Number of Decision Trees in the forest
        self.n_estimators = n_estimators

        # Maximum depth of each Decision Tree
        self.max_depth = max_depth

        # Minimum number of samples required to split a node
        self.min_samples_split = min_samples_split

        # Number of random features considered at each split
        self.max_features = max_features

        # List of trained Decision Trees
        self.trees = []


    # =========================================================
    # Bootstrap Sampling
    # =========================================================

    bootstrap_sample = bootstrap_sample

    # =========================================================
    # Training
    # =========================================================

    fit = fit


    # =========================================================
    # Prediction
    # =========================================================

    predict = predict
    predict_proba = predict_proba

### Testing the Random Forest

After implementing all components of the Random Forest, the model is trained using the training dataset. The training accuracy is then calculated by comparing the predicted class labels with the actual labels.

Although training accuracy does not measure how well the model generalises to unseen data, it provides a useful sanity check to verify that the Random Forest has been implemented correctly and is capable of learning meaningful patterns from the training data.

In [ ]:
# Create a Random Forest classifier
forest = RandomForest(
    n_estimators=100,
    max_depth=10,
    min_samples_split=2,
    max_features="sqrt"
)

# Train the Random Forest
forest.fit(
    X_train,
    y_train
)

# Predict the training data
predictions = forest.predict(
    X_train
)

# Calculate the training accuracy
accuracy = np.mean(
    predictions == y_train
)

print(f"Training Accuracy: {accuracy:.4f}")

In [ ]:
# Predict class probabilities for the first five passengers
probabilities = forest.predict_proba(
    X_train[:5]
)

print("Prediction Probabilities:")
print(probabilities)

## Hyperparameter Tuning

Hyperparameter tuning is performed to identify the Random Forest configuration that produces the highest predictive performance. Different combinations of hyperparameters are evaluated by training multiple Random Forest models and comparing their average training accuracy.

The following hyperparameters are tested:

- **Number of Trees (`n_estimators`)**: 10, 50, 100, 200
- **Maximum Tree Depth (`max_depth`)**: 3, 5, 10, None (no depth limit)
- **Minimum Samples Required to Split (`min_samples_split`)**: 2, 5, 10

Since Random Forest uses bootstrap sampling and random feature selection, the performance of a single model may vary slightly between runs. Therefore, **each hyperparameter combination is trained 10 times**, and the following statistics are recorded:

- **Average Accuracy** – Mean training accuracy across the 10 runs.
- **Standard Deviation** – Measures the consistency of the model's performance.

The final results are displayed in a table sorted in **descending order of average accuracy**. If two configurations achieve the same average accuracy, the configuration with the **lower standard deviation** is ranked higher because it produces more consistent results.

In [ ]:
# --------------------------------------------------
# Create one fixed 80/20 train-test split
# --------------------------------------------------

num_samples = len(X_train)

# Shuffle once
indices = np.random.permutation(num_samples)

split_index = int(0.8 * num_samples)

train_indices = indices[:split_index]
test_indices = indices[split_index:]

# Fixed training set
X_train_split = X_train[train_indices]
y_train_split = y_train[train_indices]

# Fixed testing set
X_test_split = X_train[test_indices]
y_test_split = y_train[test_indices]

### Note

The hyperparameter tuning experiment evaluates multiple combinations of Random Forest hyperparameters, with each configuration trained and evaluated over 10 independent runs. As a result, this section takes approximately **40 minutes** to complete.

To improve the usability of the notebook, hyperparameter tuning is **disabled by default**. The final selected hyperparameters, experimental results, and analysis are already included in this notebook.

To reproduce the hyperparameter tuning experiments, set the following flag to `True` and execute this section:

```python
RUN_HYPERPARAMETER_TUNING = True
```

Otherwise, leave the flag as `False` to skip this section and proceed directly to the final model evaluation.

In [ ]:
RUN_HYPERPARAMETER_TUNING = False
if RUN_HYPERPARAMETER_TUNING:
    # --------------------------------------------------
    # Hyperparameter values to test
    # --------------------------------------------------

    n_estimators_list = [10, 50, 100, 200]
    max_depth_list = [3, 5, 10, None]
    min_samples_split_list = [2, 5, 10]

    num_runs = 10

    results = []

    # --------------------------------------------------
    # Hyperparameter Tuning
    # --------------------------------------------------

    for n_estimators in n_estimators_list:
        for max_depth in max_depth_list:
            for min_samples_split in min_samples_split_list:

                accuracies = []

                for _ in range(num_runs):

                    forest = RandomForest(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        min_samples_split=min_samples_split,
                        max_features="sqrt"
                    )

                    # Train on the fixed 80%
                    forest.fit(
                        X_train_split,
                        y_train_split
                    )

                    # Evaluate on the fixed 20%
                    predictions = forest.predict(
                        X_test_split
                    )

                    accuracy = np.mean(
                        predictions == y_test_split
                    )

                    accuracies.append(accuracy)

                results.append({
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "min_samples_split": min_samples_split,
                    "average_accuracy": np.mean(accuracies),
                    "std_accuracy": np.std(accuracies)
                })

    # --------------------------------------------------
    # Results Table
    # --------------------------------------------------

    results_df = pd.DataFrame(results)

    results_df = results_df.sort_values(
        by=["average_accuracy", "std_accuracy"],
        ascending=[False, True]
    ).reset_index(drop=True)

    display(results_df)


### Hyperparameter Tuning - Conclusion

Several important trends were observed during hyperparameter tuning.

#### Effect of the Number of Trees

Increasing the number of trees generally improved the model's performance. Since each Decision Tree is trained on a different bootstrap sample and considers a different subset of features, adding more trees allows the Random Forest to produce more reliable predictions through majority voting.

However, the improvement gradually diminished as additional trees were added. For example, increasing the forest size from **100 trees** to **200 trees** resulted in only a **0.11 percentage point** increase in average accuracy. This illustrates the principle of **diminishing returns**—after a sufficiently large ensemble has been built, additional trees provide only marginal improvements while requiring substantially more computation.

---

#### Effect of Maximum Tree Depth

The strongest-performing models generally used a **maximum depth of 5**. Although deeper trees (`10` or `None`) are capable of learning more complex decision boundaries, they are also more prone to **overfitting** the training data. For the relatively small Titanic dataset, limiting the tree depth produced models that generalised better to unseen passengers.

---

#### Effect of Minimum Samples Required to Split

Changing `min_samples_split` had only a small influence on overall performance. Values of **2**, **5**, and **10** produced similar accuracies, indicating that the Random Forest was relatively insensitive to this parameter for the Titanic dataset.

---

#### Selecting the Final Model

The highest average accuracy was achieved using:

- **200 trees**
- **Maximum depth = 5**
- **Minimum samples split = 5**

However, the configuration using:

- **100 trees**
- **Maximum depth = 5**
- **Minimum samples split = 2**

achieved an average accuracy only **0.11 percentage points lower**.

This difference is smaller than the variation observed across repeated runs, indicating that the two models perform very similarly in practice. Considering the additional computational cost required to train 200 trees, the configuration with **100 trees**, **maximum depth of 5**, and **minimum samples split of 2** was selected as the final model.

This configuration provides an excellent balance between prediction accuracy, model stability, and computational efficiency, making it the most practical choice for this project.

## 10. Model Evaluation

After hyperparameter tuning, the following configuration was selected as the final Random Forest model:

- **Number of Trees:** 100
- **Maximum Tree Depth:** 5
- **Minimum Samples for Split:** 2
- **Maximum Features per Split:** $\sqrt{\text{Number of Features}}$

This configuration provided an excellent balance between **classification performance**, **training time**, and **computational efficiency**, making it the most practical choice for the final model.

---

To obtain reliable performance estimates, the model was evaluated over **20 independent runs**.

For each run:

- A new Random Forest was trained using bootstrap sampling.
- The model was evaluated on the **same fixed 20% testing set**.
- The reported evaluation metrics are the **average over all 20 runs**, reducing the effect of randomness introduced during training.

The following metrics were used to evaluate the model.

---

## • Accuracy

Accuracy measures the proportion of correctly classified samples.

$$
\text{Accuracy}
=
\frac{TP + TN}
{TP + TN + FP + FN}
$$

**Example**

If the model correctly classifies **160** out of **179** passengers,

$$
\text{Accuracy}
=
\frac{160}{179}
=
89.39\%
$$

---

## • Confusion Matrix

The confusion matrix provides a complete breakdown of the model's predictions.

| Actual \ Predicted | Survived | Did Not Survive |
|:-------------------|:--------:|:---------------:|
| **Survived** | True Positive (**TP**) | False Negative (**FN**) |
| **Did Not Survive** | False Positive (**FP**) | True Negative (**TN**) |

Unlike accuracy, the confusion matrix reveals **how** the model makes mistakes and forms the basis for the remaining evaluation metrics.

---

## • Precision

Precision measures how many passengers predicted as **Survived** actually survived.

$$
\text{Precision}
=
\frac{TP}
{TP + FP}
$$

A higher precision indicates **fewer false positives**.

---

## • Recall

Recall measures how many of the actual survivors were correctly identified.

$$
\text{Recall}
=
\frac{TP}
{TP + FN}
$$

A higher recall indicates **fewer false negatives**.

---

## • F1-Score

The F1-Score combines Precision and Recall into a single metric.

$$
F_1
=
2
\times
\frac{\text{Precision}\times\text{Recall}}
{\text{Precision}+\text{Recall}}
$$

The F1-Score is particularly useful when both **false positives** and **false negatives** are equally important, providing a balanced measure of classification performance.

In [ ]:
# --------------------------------------------------
# Final Model Evaluation
# --------------------------------------------------

num_runs = 20

accuracies = []
precisions = []
recalls = []
f1_scores = []

total_tp = 0
total_tn = 0
total_fp = 0
total_fn = 0

for _ in range(num_runs):

    # -------------------------------------
    # Create Random Forest
    # -------------------------------------

    forest = RandomForest(
        n_estimators=100,
        max_depth=5,
        min_samples_split=2,
        max_features="sqrt"
    )

    # -------------------------------------
    # Train
    # -------------------------------------

    forest.fit(
        X_train_split,
        y_train_split
    )

    # -------------------------------------
    # Predict
    # -------------------------------------

    predictions = forest.predict(
        X_test_split
    )

    # -------------------------------------
    # Accuracy
    # -------------------------------------

    accuracy = np.mean(
        predictions == y_test_split
    )

    accuracies.append(accuracy)

    # -------------------------------------
    # Confusion Matrix
    # -------------------------------------

    tp = np.sum(
        (predictions == 1) &
        (y_test_split == 1)
    )

    tn = np.sum(
        (predictions == 0) &
        (y_test_split == 0)
    )

    fp = np.sum(
        (predictions == 1) &
        (y_test_split == 0)
    )

    fn = np.sum(
        (predictions == 0) &
        (y_test_split == 1)
    )

    total_tp += tp
    total_tn += tn
    total_fp += fp
    total_fn += fn

    # -------------------------------------
    # Precision
    # -------------------------------------

    if (tp + fp) == 0:
        precision = 0
    else:
        precision = tp / (tp + fp)

    precisions.append(precision)

    # -------------------------------------
    # Recall
    # -------------------------------------

    if (tp + fn) == 0:
        recall = 0
    else:
        recall = tp / (tp + fn)

    recalls.append(recall)

    # -------------------------------------
    # F1 Score
    # -------------------------------------

    if (precision + recall) == 0:
        f1 = 0
    else:
        f1 = (
            2 * precision * recall
        ) / (
            precision + recall
        )

    f1_scores.append(f1)

# --------------------------------------------------
# Average Metrics
# --------------------------------------------------

average_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

average_precision = np.mean(precisions)
average_recall = np.mean(recalls)
average_f1 = np.mean(f1_scores)

average_tp = total_tp / num_runs
average_tn = total_tn / num_runs
average_fp = total_fp / num_runs
average_fn = total_fn / num_runs

# --------------------------------------------------
# Results
# --------------------------------------------------

print("========== Final Model Evaluation ==========\n")

print(f"Average Accuracy          : {average_accuracy:.4f}")
print(f"Accuracy Std. Deviation   : {std_accuracy:.4f}")

print(f"\nAverage Precision         : {average_precision:.4f}")
print(f"Average Recall            : {average_recall:.4f}")
print(f"Average F1 Score          : {average_f1:.4f}")

print("\n========== Average Confusion Matrix ==========\n")

print(f"                 Predicted")
print(f"               1         0")
print(f"Actual 1   {average_tp:7.2f}   {average_fn:7.2f}")
print(f"Actual 0   {average_fp:7.2f}   {average_tn:7.2f}")

### Conclusion

The final Random Forest model achieved an **average accuracy of 87.57%**, with a **standard deviation of only 0.61%** over 20 independent runs. The low standard deviation indicates that the model produces **stable and consistent predictions**, demonstrating that the ensemble is not overly sensitive to the randomness introduced by bootstrap sampling.

The **average confusion matrix** further illustrates the model's performance:

| Actual \ Predicted | Survived | Did Not Survive |
|:-------------------|---------:|----------------:|
| **Survived** | 46.20 | 17.80 |
| **Did Not Survive** | 4.45 | 110.55 |

The model produced relatively **few False Positives (4.45)** while correctly identifying the majority of non-survivors. However, it missed a portion of the actual survivors (**17.80 False Negatives**), indicating that the model is slightly conservative when predicting survival.

This behaviour is reflected in the evaluation metrics:

- **Precision:** **91.25%**
- **Recall:** **72.19%**
- **F1-Score:** **80.58%**

The **high Precision** shows that when the model predicts a passenger will survive, the prediction is correct in most cases. In contrast, the lower Recall indicates that some actual survivors are incorrectly classified as non-survivors. Consequently, the model prioritises **prediction reliability** over identifying every survivor.

The **F1-Score of 80.58%** demonstrates a strong balance between Precision and Recall, indicating that the classifier performs well overall while maintaining reliable positive predictions.

Overall, the implemented Random Forest successfully captures the underlying patterns within the Titanic dataset and demonstrates that a manually implemented ensemble of Decision Trees can achieve **accurate, stable, and reliable classification performance** without relying on external machine learning libraries.

## 11. Feature Importance Analysis

Not every feature contributes equally to predicting passenger survival. Some features consistently produce **better dataset splits**, making them more valuable to the model.

To measure the contribution of each feature, **Information Gain** based on **Gini Impurity** is used.

---

## Gini Impurity

Gini Impurity measures how mixed the classes are within a node.

$$
G
=
1
-
\sum_{i=1}^{c}p_i^2
$$

where

- $p_i$ = Probability of class $i$
- $c$ = Number of classes

A **lower Gini Impurity** indicates a purer node.

---

## Information Gain

Information Gain measures the reduction in Gini Impurity after splitting the dataset.

$$
\text{Information Gain}
=
G(\text{Parent})
-
\left(
\frac{N_L}{N}G(\text{Left})
+
\frac{N_R}{N}G(\text{Right})
\right)
$$

where

- $G(\text{Parent})$ = Gini Impurity before splitting
- $G(\text{Left})$ = Gini Impurity of the left child
- $G(\text{Right})$ = Gini Impurity of the right child

A larger Information Gain indicates a **better split** and therefore a **more informative feature**.

---

## Example

Suppose a node is evaluated using four different features.

| Feature | Information Gain |
|:--------|-----------------:|
| Sex | **0.42** |
| Pclass | 0.21 |
| Age | 0.18 |
| Fare | 0.09 |

Since **Sex** produces the highest Information Gain, it is selected for the split.

The score **0.42** is then credited to the feature **Sex**.

---

## Decision Tree Feature Importance

During tree construction, every selected split contributes its Information Gain to the feature responsible for that split.

For example,

```text
Sex  → +0.42
Age  → +0.18
Fare → +0.09
```

After the entire tree is constructed, each feature has accumulated a total Information Gain.

---

## Random Forest Feature Importance

A Random Forest consists of multiple Decision Trees.

The final importance of each feature is obtained by summing its Information Gain across **every split in every tree**.

$$
\text{Feature Importance}_j
=
\sum_{t=1}^{T}
\sum_{\text{Splits}}
\text{Information Gain}
$$

where

- $T$ = Number of Decision Trees
- $j$ = Feature being evaluated

Features that consistently produce informative splits across multiple trees accumulate larger scores and are therefore considered more important.

---

## Normalisation

The accumulated scores are converted into relative importance values by dividing each feature's score by the total Information Gain.

$$
\text{Normalised Importance}_j
=
\frac{\text{Importance}_j}
{\sum_{k=1}^{m}\text{Importance}_k}
$$

where $m$ is the total number of features.

After normalisation,

$$
\sum_{j=1}^{m}
\text{Normalised Importance}_j
=
1
$$

This allows the importance of different features to be compared directly and expressed as percentages.

In [ ]:
# --------------------------------------------------
# Weighted Feature Importance
# --------------------------------------------------

feature_importance = np.zeros(X_train_split.shape[1])

total_samples = len(X_train_split)


def calculate_feature_importance(node):

    if node is None:
        return

    if node.value is None:

        # Weighted Information Gain
        feature_importance[node.feature] += (

            node.information_gain

            *

            (node.num_samples / total_samples)

        )

        calculate_feature_importance(node.left)
        calculate_feature_importance(node.right)


# Traverse every tree
for tree in forest.trees:

    calculate_feature_importance(tree.root)


# --------------------------------------------------
# Normalise
# --------------------------------------------------

feature_importance = (

    feature_importance

    /

    np.sum(feature_importance)

)


# --------------------------------------------------
# Display
# --------------------------------------------------

importance_df = pd.DataFrame({

    "Feature": train_df.drop(
        columns=["Survived"]
    ).columns,

    "Importance": feature_importance,

    "Importance (%)":

        feature_importance * 100

})

importance_df = importance_df.sort_values(

    by="Importance",

    ascending=False

).reset_index(drop=True)

display(importance_df)

## Conclusion

This project demonstrates the complete implementation of a **Random Forest Classifier from scratch**, beginning with raw data exploration and ending with model interpretation. Every major component—including Decision Tree construction, bootstrap sampling, Information Gain calculation, random feature selection, hyperparameter tuning, and feature importance analysis—was implemented manually to understand the underlying algorithms rather than relying on pre-built machine learning libraries.

The final model was trained using the following hyperparameters:

- **Number of Trees:** 100
- **Maximum Tree Depth:** 5
- **Minimum Samples for Split:** 2
- **Maximum Features per Split:** √(Number of Features)

This configuration was selected because it achieved excellent predictive performance while remaining computationally efficient.

---

# Understanding the Random Forest

A **Random Forest** is an **ensemble learning algorithm**, meaning that instead of relying on a single Decision Tree, it combines the predictions of many Decision Trees to produce a more reliable final prediction.

The training process consists of four major stages.

---

## 1. Bootstrap Sampling

For every Decision Tree, a new training dataset is created by randomly sampling passengers **with replacement** from the original training set.

Because sampling is performed with replacement:

- Some passengers may appear multiple times.
- Some passengers may not appear at all.

As a result, every tree learns from a slightly different dataset.

---

## 2. Random Feature Selection

Rather than evaluating every feature at every split, only a random subset of features is considered.

For this project,

$$
\text{Maximum Features}
=
\sqrt{13}
\approx
4
$$

Therefore, every node randomly evaluates only **4 features** before selecting the best split.

This forces different trees to learn different decision boundaries and greatly increases the diversity of the forest.

---

## 3. Decision Tree Construction

Each Decision Tree recursively partitions the dataset.

For every candidate split:

- The dataset is divided into two child nodes.
- The impurity before the split is calculated.
- The impurity after the split is calculated.
- The reduction in impurity is measured using **Information Gain**.

Information Gain is calculated as

$$
\text{Information Gain}
=
G_{\text{parent}}
-
G_{\text{weighted children}}
$$

where the **Gini Impurity** is

$$
G
=
1-\sum p_i^2
$$

A split producing the highest Information Gain is selected.

The process continues until one of the stopping conditions is reached:

- Pure node
- Maximum depth reached
- Too few samples remaining

---

## 4. Majority Voting

Once all Decision Trees have been trained, every tree independently predicts the class of a passenger.

The final Random Forest prediction is determined using **majority voting**.

Example

| Tree | Prediction |
|------|------------|
| Tree 1 | Survived |
| Tree 2 | Did Not Survive |
| Tree 3 | Survived |
| Tree 4 | Survived |
| Tree 5 | Did Not Survive |

Final Prediction:

**Survived (3 votes vs 2 votes)**

Since every tree has learned slightly different patterns, incorrect predictions made by individual trees are often cancelled by the majority vote.

---

# Interpreting the Model

Unlike many machine learning models, a Random Forest allows us to understand **why** predictions are made.

Every split inside every Decision Tree is chosen using **Information Gain**.

Whenever a feature is selected, it contributes a certain amount of Information Gain by reducing the uncertainty of the data.

By accumulating the Information Gain contributed by every feature across all Decision Trees, we obtain a Feature Importance score.

Features contributing larger Information Gain more frequently are considered more influential.

---

# What Influenced Survival?

The Feature Importance Analysis produced the following ranking.

| Rank | Feature | Importance |
|------|---------|-----------:|
| 1 | Title_Mr | 23.65% |
| 2 | Sex | 21.37% |
| 3 | Fare | 12.17% |
| 4 | Pclass | 9.84% |
| 5 | Title_Miss | 7.97% |
| 6 | FamilySize | 7.07% |
| 7 | Age | 6.58% |

Several important patterns emerge.

### Adult males

The most influential feature was **Title_Mr**, followed immediately by **Sex**.

This indicates that being an adult male was one of the strongest indicators of non-survival.

The model naturally separates titles because they indirectly capture:

- Gender
- Age
- Social status

rather than relying only on the Sex column.

---

### Passenger Class and Fare

Both **Fare** and **Passenger Class** ranked among the most important predictors.

Passengers paying higher fares generally belonged to higher passenger classes, giving them better access to lifeboats and significantly increasing their probability of survival.

---

### Family Size

Family Size contributed more than expected.

Passengers travelling with a small family generally survived more often than passengers travelling completely alone or within very large families.

The model therefore considered Family Size to be a meaningful predictor.

---

### Age

Although younger passengers generally exhibited higher survival rates, the age distributions of survivors and non-survivors overlapped considerably.

Consequently, Age contributed useful information but ranked below several social features.

---

### Embarked

Features related to **Embarked** contributed very little to the final model.

This suggests that the port from which a passenger boarded the Titanic had little influence on survival once stronger predictors such as passenger class, gender and ticket fare were considered.

---

# Supporting Evidence from Data Analysis

The Feature Importance results closely matched the earlier exploratory analysis.

The visualisations showed that:

- First-class passengers survived much more frequently than third-class passengers.
- Female passengers had substantially higher survival rates than males.
- Younger passengers generally had a better chance of survival.
- Higher fares were strongly associated with higher passenger classes.

The missing-value analysis also justified the preprocessing strategy adopted throughout the project.

- **Cabin** contained over **77%** missing values and was therefore removed.
- **Age** contained approximately **20%** missing values and was imputed using hierarchical grouping.
- **Embarked** contained only two missing values and was also imputed using hierarchical grouping.

These preprocessing decisions preserved as much information as possible while ensuring that the final dataset remained complete.

---

# Model Generalisation

Hyperparameter tuning demonstrated that increasing the number of trees beyond **100** produced only marginal improvements while substantially increasing computation time.

Similarly, limiting the maximum tree depth to **5** prevented individual Decision Trees from memorising the training data while still allowing them to capture meaningful decision boundaries.

Repeated evaluation over **20 independent runs** produced extremely consistent results, indicating that the model behaves reliably despite the randomness introduced through bootstrap sampling.

Overall, there is **no strong evidence of either severe overfitting or underfitting**.

Instead, the model successfully balances:

- Predictive performance
- Computational efficiency
- Interpretability

---

# Final Remarks

This project demonstrates the complete implementation of a Random Forest classifier from scratch while following the full machine learning workflow.

Beginning with exploratory data analysis and preprocessing, progressing through manual Decision Tree construction, Random Forest implementation, hyperparameter tuning, model evaluation and feature importance analysis, every stage was implemented independently without relying on pre-built machine learning algorithms.

The final model not only achieves strong predictive performance but also provides meaningful insight into the historical factors that most influenced passenger survival, making it both accurate and highly interpretable.

# Experimental Modification Proposals

Although the standard Random Forest achieved strong and consistent performance, several modifications could be explored to further improve generalisation and reduce overfitting.

These ideas were identified during development but were not incorporated into the final implementation to preserve the standard Random Forest algorithm and ensure a fair comparison with the baseline model.

---

## Proposal 1 — Dynamic Random Feature Selection

### Standard Random Forest

At every node, the algorithm considers

$$
\sqrt{p}
$$

randomly selected features,

where

- \(p\) = total number of features.

For this project,

$$
\sqrt{13}\approx3
$$

candidate features were evaluated at every split.

---

### Proposed Modification

Instead of always selecting exactly \(\sqrt{p}\) features, randomly select

$$
k\sim U(\sqrt{p},\,p)
$$

where

$$
\sqrt{p}\le k\le p.
$$

This allows different nodes to evaluate different numbers of candidate features.

Why lower bound by $\sqrt{p}$ ?

A lower bound of $\sqrt{p}$ was chosen instead of 1 because selecting too few candidate features can make each Decision Tree too random, reducing its ability to find informative splits.

If only one feature is randomly selected at a node, the algorithm has no choice but to split using that feature, even if it provides very little Information Gain.

Example

- Node 1 evaluates 4 features.
- Node 2 evaluates 8 features.
- Node 3 evaluates 11 features.

---

### Potential Benefits

- Increased diversity between Decision Trees.
- Some nodes gain access to stronger candidate splits.
- May improve overall predictive performance on certain datasets.

---

### Potential Drawbacks

- Larger feature subsets make different trees more similar.
- Reduced diversity may weaken the ensemble effect.
- Additional computation is required for larger feature subsets.

---

## Proposal 2 — Variable Bootstrap Sample Size

### Standard Random Forest

Every Decision Tree is trained using a bootstrap sample containing

$$
N
$$

training samples,

where \(N\) is the size of the original training set.

---

### Proposed Modification

Instead of fixing the bootstrap size, randomly sample

$$
m\sim U(0.6N,\;N)
$$

for every Decision Tree.

Example

Tree 1 → 560 passengers

Tree 2 → 730 passengers

Tree 3 → 891 passengers

---

### Potential Benefits

- Greater variation between Decision Trees.
- May reduce correlation between trees.
- Could improve ensemble robustness.

---

### Potential Drawbacks

- Very small bootstrap samples may produce weak Decision Trees.
- Additional randomness does not necessarily improve prediction accuracy.
- Performance improvements would require empirical validation.

---

## Overfitting and Underfitting Considerations

### Potential Sources of Overfitting

- Excessive tree depth.
- Very small leaf nodes.
- Highly correlated Decision Trees.

Mitigation used in this project:

- Maximum Tree Depth = 5.
- Random feature selection at every split.
- Bootstrap sampling.
- Majority voting across 100 Decision Trees.

---

### Potential Sources of Underfitting

- Tree depth too small.
- Too few Decision Trees.
- Insufficient candidate features during splitting.

Increasing these parameters may improve learning but could also increase overfitting.

---

## Future Work

Future experiments could compare these proposed modifications directly against the standard Random Forest using identical training and testing splits.

Performance metrics such as Accuracy, Precision, Recall, F1-Score and Feature Importance could then be analysed to determine whether the additional randomness improves the model or simply increases variance.